# LambdaLLG Seed Relaxation Examples

This notebook initializes and relaxes the current seed helpers in seven cases:

1. A single 1D Neel wall.
2. A single 1D Bloch wall.
3. A single 2D Bloch skyrmion.
4. A single 2D Neel skyrmion.
5. Two 1D domain walls in the same seed.
6. Two 2D domain walls in the same seed.
7. Two 2D skyrmions in the same seed.

All seeds are built additively on top of uniform backgrounds and normalized before relaxation.


In [ ]:
using Pkg; Pkg.activate("../")


In [ ]:
using LambdaLLG
using LinearAlgebra
using Statistics
import PyPlot as plt

plt.rcParams["figure.dpi"] = 140


## Helpers

The 1D plots use the blue, green, red convention for the x, y, z spin components. The 2D texture plots use `pcolormesh` for `m_z` together with in-plane quivers.


In [ ]:
function plot_chain!(ax, spins; title="")
    x = collect(axes(spins, 2))
    ax.plot(x, vec(spins[1, :]), color="blue", label="m_x")
    ax.plot(x, vec(spins[2, :]), color="green", label="m_y")
    ax.plot(x, vec(spins[3, :]), color="red", label="m_z")
    ax.set_ylim(-1.05, 1.05)
    ax.set_xlabel("site")
    ax.set_ylabel("spin")
    ax.set_title(title)
    ax.grid(true, alpha=0.25)
    return ax
end

function plot_texture!(ax, spins; title="", stride=5)
    Nx, Ny = size(spins, 2), size(spins, 3)
    mz = [spins[3, i, j] for j in 1:Ny, i in 1:Nx]
    im = ax.pcolormesh(1:Nx, 1:Ny, mz; cmap="RdBu_r", shading="auto", vmin=-1, vmax=1)

    xs = collect(1:stride:Nx)
    ys = collect(1:stride:Ny)
    X = [i for j in ys, i in xs]
    Y = [j for j in ys, i in xs]
    U = [spins[1, i, j] for j in ys, i in xs]
    V = [spins[2, i, j] for j in ys, i in xs]
    ax.quiver(X, Y, U, V; color="black", pivot="mid", scale=28)

    ax.set_aspect("equal")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(title)
    return im
end

function wall_site(spins)
    idx = argmin(abs.(spins[3, :]))
    return (index=idx, spin=spins[:, idx])
end

function core_site(spins)
    mz = spins[3, :, :]
    idx = argmin(mz)
    cart = CartesianIndices(mz)[idx]
    i, j = Tuple(cart)
    return (i=i, j=j, spin=spins[:, i, j])
end


## 1D Domain Walls

These examples use the same easy-axis 1D model and compare a Neel wall, a Bloch wall, and a two-wall seed made by painting twice onto the same uniform background.


In [ ]:
Nx1 = 181
J1 = 1.0
K1 = (0.0, 0.0, 0.35)
alpha1 = 0.25
t_relax1 = (0.0, 120.0)
p1 = LLGParams1D(Nx1, J1, K1, (0.0, 0.0, 0.0), alpha1)

s_neel0 = uniform_seed1D(Nx1; direction=(0.0, 0.0, 1.0))
paint_domain_wall1D!(s_neel0; center=(Nx1 + 1) / 2, width=6.0, domain_direction=(0.0, 0.0, 1.0), wall_normal=(1.0, 0.0, 0.0))
normalize_spins!(s_neel0)
sol_neel = evolve1D(copy(s_neel0), t_relax1, p1; reltol=1e-6, abstol=1e-6)
s_neel = reshape(copy(sol_neel.u[end]), 3, Nx1)

s_bloch0 = uniform_seed1D(Nx1; direction=(0.0, 0.0, 1.0))
paint_domain_wall1D!(s_bloch0; center=(Nx1 + 1) / 2, width=6.0, domain_direction=(0.0, 0.0, 1.0), wall_normal=(0.0, 1.0, 0.0))
normalize_spins!(s_bloch0)
sol_bloch = evolve1D(copy(s_bloch0), t_relax1, p1; reltol=1e-6, abstol=1e-6)
s_bloch = reshape(copy(sol_bloch.u[end]), 3, Nx1)

s_two_walls0 = uniform_seed1D(Nx1; direction=(0.0, 0.0, 1.0))
paint_domain_wall1D!(s_two_walls0; center=61.0, width=6.0, domain_direction=(0.0, 0.0, 1.0), wall_normal=(1.0, 0.0, 0.0))
paint_domain_wall1D!(s_two_walls0; center=121.0, width=6.0, domain_direction=(0.0, 0.0, -1.0), wall_normal=(0.0, 1.0, 0.0))
normalize_spins!(s_two_walls0)
sol_two_walls = evolve1D(copy(s_two_walls0), t_relax1, p1; reltol=1e-6, abstol=1e-6)
s_two_walls = reshape(copy(sol_two_walls.u[end]), 3, Nx1)

display((neel_wall=wall_site(s_neel), bloch_wall=wall_site(s_bloch), pair_midpoint=wall_site(s_two_walls)))

fig1, ax1 = plt.subplots(3, 2, figsize=(12, 10), sharex=true, sharey=true)
plot_chain!(ax1[1, 1], s_neel0; title="1D Neel seed")
plot_chain!(ax1[1, 2], s_neel; title="1D Neel relaxed")
plot_chain!(ax1[2, 1], s_bloch0; title="1D Bloch seed")
plot_chain!(ax1[2, 2], s_bloch; title="1D Bloch relaxed")
plot_chain!(ax1[3, 1], s_two_walls0; title="Two 1D walls seed")
plot_chain!(ax1[3, 2], s_two_walls; title="Two 1D walls relaxed")
ax1[1, 1].legend(loc="lower right")
fig1


## 2D Textures

The Bloch skyrmion is relaxed with bulk DMI, the Neel skyrmion and the two-skyrmion seed are relaxed with interfacial DMI, and the two-wall seed is relaxed in the easy-axis model without DMI.


In [ ]:
Nx2 = 101
Ny2 = 101
J2 = 1.0
alpha2 = 0.3
t_relax2 = (0.0, 120.0)

K_bloch = (0.0, 0.0, 0.45)
D_bloch = ((0.25, 0.0, 0.0), (0.0, 0.25, 0.0))
p_bloch = LLGParams2D(Nx2, Ny2, J2, K_bloch, (0.0, 0.0, 0.0), alpha2; D=D_bloch)
s_bloch_sk0 = uniform_seed2D(Nx2, Ny2; direction=(0.0, 0.0, 1.0))
paint_skyrmion2D!(s_bloch_sk0; center=((Nx2 + 1) / 2, (Ny2 + 1) / 2), radius=10.0, width=2.8, center_direction=-1, helicity=pi / 2, vorticity=1.0)
normalize_spins!(s_bloch_sk0)
sol_bloch_sk = evolve2D(copy(s_bloch_sk0), t_relax2, p_bloch; reltol=1e-6, abstol=1e-6)
s_bloch_sk = reshape(copy(sol_bloch_sk.u[end]), 3, Nx2, Ny2)

K_neel = (0.0, 0.0, 0.15)
D_neel = ((0.0, 0.2, 0.0), (-0.2, 0.0, 0.0))
p_neel = LLGParams2D(Nx2, Ny2, J2, K_neel, (0.0, 0.0, 0.0), alpha2; D=D_neel)
s_neel_sk0 = uniform_seed2D(Nx2, Ny2; direction=(0.0, 0.0, 1.0))
paint_skyrmion2D!(s_neel_sk0; center=((Nx2 + 1) / 2, (Ny2 + 1) / 2), radius=10.0, width=2.8, center_direction=-1, helicity=0.0, vorticity=1.0)
normalize_spins!(s_neel_sk0)
sol_neel_sk = evolve2D(copy(s_neel_sk0), t_relax2, p_neel; reltol=1e-6, abstol=1e-6)
s_neel_sk = reshape(copy(sol_neel_sk.u[end]), 3, Nx2, Ny2)

K_walls2 = (0.0, 0.0, 0.35)
p_walls2 = LLGParams2D(Nx2, Ny2, J2, K_walls2, (0.0, 0.0, 0.0), alpha2)
s_two_fronts0 = uniform_seed2D(Nx2, Ny2; direction=(0.0, 0.0, 1.0))
paint_domain_wall2D!(s_two_fronts0; point=(51.0, 33.0), slope=0.0, width=4.0, domain_direction=(0.0, 0.0, 1.0), wall_normal=(1.0, 0.0, 0.0))
paint_domain_wall2D!(s_two_fronts0; point=(51.0, 69.0), slope=0.0, width=4.0, domain_direction=(0.0, 0.0, -1.0), wall_normal=(0.0, 1.0, 0.0))
normalize_spins!(s_two_fronts0)
sol_two_fronts = evolve2D(copy(s_two_fronts0), t_relax2, p_walls2; reltol=1e-6, abstol=1e-6)
s_two_fronts = reshape(copy(sol_two_fronts.u[end]), 3, Nx2, Ny2)

s_two_sk0 = uniform_seed2D(Nx2, Ny2; direction=(0.0, 0.0, 1.0))
paint_skyrmion2D!(s_two_sk0; center=(33.0, 51.0), radius=9.0, width=2.8, center_direction=-1, helicity=0.0, vorticity=1.0)
paint_skyrmion2D!(s_two_sk0; center=(69.0, 51.0), radius=9.0, width=2.8, center_direction=-1, helicity=0.0, vorticity=1.0)
normalize_spins!(s_two_sk0)
sol_two_sk = evolve2D(copy(s_two_sk0), t_relax2, p_neel; reltol=1e-6, abstol=1e-6)
s_two_sk = reshape(copy(sol_two_sk.u[end]), 3, Nx2, Ny2)

display((bloch_core=core_site(s_bloch_sk), neel_core=core_site(s_neel_sk), pair_core=core_site(s_two_sk)))

fig2, ax2 = plt.subplots(4, 2, figsize=(10, 18))
im = plot_texture!(ax2[1, 1], s_bloch_sk0; title="Bloch skyrmion seed", stride=5)
plot_texture!(ax2[1, 2], s_bloch_sk; title="Bloch skyrmion relaxed", stride=5)
plot_texture!(ax2[2, 1], s_neel_sk0; title="Neel skyrmion seed", stride=5)
plot_texture!(ax2[2, 2], s_neel_sk; title="Neel skyrmion relaxed", stride=5)
plot_texture!(ax2[3, 1], s_two_fronts0; title="Two 2D walls seed", stride=5)
plot_texture!(ax2[3, 2], s_two_fronts; title="Two 2D walls relaxed", stride=5)
plot_texture!(ax2[4, 1], s_two_sk0; title="Two skyrmions seed", stride=5)
plot_texture!(ax2[4, 2], s_two_sk; title="Two skyrmions relaxed", stride=5)
fig2.colorbar(im, ax=ax2[:, :], shrink=0.9, label="m_z")
fig2
